# Customer Churn — Data Quality & Exploratory Analysis

**Purpose:** Create a reproducible, analysis-ready customer dataset and document data-quality decisions and initial churn patterns.

**Dataset:** `customer_churn_sample.csv` (synthetic sample supplied for the task).  
**Reproducibility:** Keep the source CSV in the same folder as this notebook, then run all cells from top to bottom.

> This is a small teaching sample (15 rows), so patterns are descriptive and should not be treated as statistically conclusive or causal.


## 1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Change the filename only if your source CSV has a different name.
df = pd.read_csv("customer_churn_sample.csv")
print("Shape:", df.shape)
display(df.head())
display(df.info())


## 2. Data profiling: types, missingness, duplicates and categories

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(profile)
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerID values:", df["CustomerID"].duplicated().sum())
for col in df.select_dtypes(include="object").columns:
    print(f"\\n{col}:")
    print(df[col].value_counts(dropna=False))


## 3. Clean with documented assumptions

**Decisions for this supplied sample**
- Trim leading/trailing whitespace in text fields; do not change category labels otherwise.
- Convert numeric fields explicitly and preserve their numeric meaning.
- No missing values or duplicate rows/IDs were found, so no imputation or row deletion is justified.
- No IQR outliers were flagged in the numeric columns. The IQR rule is a screening flag, not proof of an error.
- Check the business arithmetic `TotalCharges ≈ MonthlyCharges × TenureMonths`. All rows reconcile exactly to two decimal places in this sample.
- Retain all 15 records. Do not remove valid extreme values without business evidence.


In [ ]:
clean = df.copy()
text_cols = clean.select_dtypes(include="object").columns
for col in text_cols:
    clean[col] = clean[col].astype("string").str.strip()

for col in ["Age", "TenureMonths", "SupportTickets"]:
    clean[col] = pd.to_numeric(clean[col], errors="raise").astype("int64")
for col in ["MonthlyCharges", "TotalCharges"]:
    clean[col] = pd.to_numeric(clean[col], errors="raise").round(2)

# Validation checks
assert clean["CustomerID"].notna().all()
assert clean["CustomerID"].is_unique
assert clean.isna().sum().sum() == 0
assert clean["Age"].between(0, 120).all()
assert (clean["TenureMonths"] >= 0).all()
assert (clean["SupportTickets"] >= 0).all()
assert set(clean["Churn"].dropna().unique()).issubset({"Yes", "No"})
charge_diff = (clean["TotalCharges"] - clean["MonthlyCharges"] * clean["TenureMonths"]).abs()
assert (charge_diff <= 0.01).all()

clean.to_csv("cleaned_customer_churn.csv", index=False)
print("Validation passed. Cleaned shape:", clean.shape)


## 4. Summary statistics and outlier screening

In [ ]:
display(clean.describe(include="all").T)

numeric_cols = clean.select_dtypes(include=np.number).columns
outlier_summary = []
for col in numeric_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((clean[col] < lower) | (clean[col] > upper)).sum()
    outlier_summary.append({
        "column": col, "lower_IQR_bound": lower,
        "upper_IQR_bound": upper, "outlier_count": int(count)
    })
display(pd.DataFrame(outlier_summary))


## 5. Exploratory analysis: churn and customer patterns

In [ ]:
churn_rate = (clean["Churn"].eq("Yes").mean() * 100)
print(f"Overall churn rate: {churn_rate:.1f}%")
display(clean["Churn"].value_counts().rename("customers").to_frame())
display(pd.crosstab(clean["ContractType"], clean["Churn"], margins=True))
display(clean.groupby("Churn")[["TenureMonths", "SupportTickets", "MonthlyCharges"]].mean().round(2))

# Churn rate by contract
contract_churn = pd.crosstab(clean["ContractType"], clean["Churn"], normalize="index")["Yes"] * 100
contract_churn.plot(kind="bar", title="Churn rate by contract type", ylabel="Churn rate (%)", xlabel="Contract type")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# Churn counts
clean["Churn"].value_counts().reindex(["No", "Yes"]).plot(kind="bar", title="Customer count by churn status", ylabel="Customers", xlabel="Churn")
plt.tight_layout()
plt.show()

# Support tickets vs churn
clean.boxplot(column="SupportTickets", by="Churn")
plt.title("Support tickets by churn status")
plt.suptitle("")
plt.ylabel("Support tickets")
plt.tight_layout()
plt.show()


## 6. Initial findings and limitations

- **Churn:** 7 of 15 customers churned (46.7%).
- **Contract pattern:** In this sample, all Month-to-Month customers churned; none of the One Year or Two Year customers churned.
- **Support:** Average support tickets were 4.29 for churned customers versus 1.00 for retained customers.
- **Tenure:** Average tenure was 7.00 months for churned customers versus 29.12 months for retained customers.

**Interpretation:** The sample suggests that short tenure, month-to-month contracts and higher support-ticket counts are associated with churn. These are descriptive associations only.

**Limitations:** Only 15 synthetic records are available. The contract/churn pattern may be an artifact of the sample. No causal claim, predictive model, or population-level conclusion is justified. A larger, representative dataset and time-aware validation would be needed.
